# Step 8: Gradient boosting with peak + history features

**Goal:** beat the published Deep PPG result (7.65 bpm, leave-one-subject-out) with classic ML.

1. Features (`ppghr.features.build_features`): PPG + motion spectra, the 5 tallest PPG peaks and 3 tallest motion peaks, the last 30 s of peak history, and running averages of the PPG spectrum. 646 numbers per window. Only the **past** is used.
2. Model (`ppghr.models.make_booster`): gradient-boosted trees trained to minimize absolute error. scikit-learn's automatic early stopping is **off**, because it uses a random 10% of windows (overlapping twins = leak).
3. Snapping: move each guess onto the nearest real PPG peak within 10 bpm.
4. Honest score: **nested** leave-one-subject-out. For each test person, an inner competition among the other 14 people picks the feature set, tree count (300 or 1000) and snapping. The test person never influences any choice.

## 1. Run the experiments

This runs `scripts/run_step8.py` inside the notebook. It takes about 15-20 minutes. Finished pieces are cached in `data/cache/step8/`, so re-running is instant.

In [ ]:
%run ../scripts/run_step8.py

## 2. Results

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ppghr.data import PROCESSED_DIR, PROJECT_DIR
from ppghr.baseline import estimate_hr_fft
from ppghr.windows import load_all_windows

summary = json.load(open(PROJECT_DIR / "results" / "step8_summary.json"))
saved = np.load(PROCESSED_DIR.parent / "step8_predictions.npz")

print(f"Honest (nested) leave-one-subject-out MAE: {summary['nested_loso_mae']} bpm")
print(f"Within 5 bpm: {summary['within_5_bpm_percent']}% of windows\n")
pd.DataFrame(summary["exploration_loso_300_trees"]).T.rename_axis("Exploration (plain LOSO, 300 trees)")

In [ ]:
choices = pd.DataFrame(summary["nested_choices"]).T
choices["MAE (bpm)"] = pd.Series(summary["nested_per_subject_mae"])
choices

In [ ]:
scores = pd.Series({
    "FFT baseline (Step 6)": 17.9,
    "SpaMa (published)": 15.56,
    "KNN (Step 7)": 13.1,
    "SpaMaPlus (published)": 11.06,
    "Deep PPG CNN (published)": 7.65,
    "Gradient boosting, nested (Step 8)": summary["nested_loso_mae"],
}).sort_values()
colors = ["darkorange" if "Step 8" in name else "steelblue" for name in scores.index]
scores.plot.barh(figsize=(8, 4), color=colors)
plt.xlabel("MAE (bpm), lower is better")
plt.title("Leave-one-subject-out heart-rate error on PPG-DaLiA")
plt.tight_layout()
plt.show()

In [ ]:
# Where did the improvement come from? Baseline vs. Step 8, by activity
from ppghr.data import ACTIVITY_NAMES
data = load_all_windows()
baseline_error = np.abs(estimate_hr_fft(data["ppg"]) - data["hr"])
step8_error = np.abs(saved["predictions"] - saved["hr"])
by_activity = pd.DataFrame({
    "FFT baseline": pd.Series(baseline_error).groupby(saved["activity"]).mean(),
    "Step 8": pd.Series(step8_error).groupby(saved["activity"]).mean(),
}).rename(index=ACTIVITY_NAMES).round(1).sort_values("Step 8")
by_activity.plot.bar(figsize=(10, 4))
plt.ylabel("MAE (bpm)")
plt.title("Error by activity")
plt.tight_layout()
plt.show()
by_activity